In [0]:
df_categoria = spark.read.format("delta").table("bronze.categoria")
df_marca     = spark.read.format("delta").table("bronze.marca")
df_modelo    = spark.read.format("delta").table("bronze.modelo")
df_estado    = spark.read.format("delta").table("bronze.estado")
df_cidade    = spark.read.format("delta").table("bronze.cidade")
df_agencia   = spark.read.format("delta").table("bronze.agencia")
df_cliente   = spark.read.format("delta").table("bronze.cliente")
df_carro     = spark.read.format("delta").table("bronze.carro")
df_reserva   = spark.read.format("delta").table("bronze.reserva")
df_pagamento = spark.read.format("delta").table("bronze.pagamento")

Aplicar Data Quality (renomear colunas e padronizar):

In [0]:
from pyspark.sql import functions as F

def aplicar_data_quality(src_fqn: str, dest_fqn: str = None):
    """
    Lê uma tabela do Bronze, padroniza os nomes das colunas
    (maiúsculo, expande siglas), remove colunas de auditoria do Bronze
    e salva no Silver com novas colunas de auditoria.
    """
    dest_fqn = dest_fqn or src_fqn

    df = spark.read.format("delta").table(src_fqn)

    # Renomeia colunas para maiúsculo e expande siglas
    def renomear(col):
        n = col.upper()
        n = n.replace("CD_", "CODIGO_")
        n = n.replace("VL_", "VALOR_")
        n = n.replace("DT_", "DATA_")
        n = n.replace("NM_", "NOME_")
        n = n.replace("DS_", "DESCRICAO_")
        n = n.replace("NR_", "NUMERO_")
        return n

    novas_colunas = [renomear(c) for c in df.columns]
    df = df.toDF(*novas_colunas)

    # Remove colunas do Bronze
    for col in ["DATA_HORA_BRONZE", "NOME_ARQUIVO"]:
        if col in df.columns:
            df = df.drop(col)

    # Adiciona colunas de auditoria do Silver
    df = df.withColumn("NOME_TABELA_BRONZE", F.lit(src_fqn)) \
           .withColumn("DATA_HORA_SILVER", F.current_timestamp())

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dest_fqn)
    return dest_fqn

Aplicar em todas as tabelas:

In [0]:
aplicar_data_quality("bronze.categoria",  "silver.categoria")
aplicar_data_quality("bronze.marca",      "silver.marca")
aplicar_data_quality("bronze.modelo",     "silver.modelo")
aplicar_data_quality("bronze.estado",     "silver.estado")
aplicar_data_quality("bronze.cidade",     "silver.cidade")
aplicar_data_quality("bronze.agencia",    "silver.agencia")
aplicar_data_quality("bronze.cliente",    "silver.cliente")
aplicar_data_quality("bronze.carro",      "silver.carro")
aplicar_data_quality("bronze.reserva",    "silver.reserva")
aplicar_data_quality("bronze.pagamento",  "silver.pagamento")

'silver.pagamento'

In [0]:
%sql
SHOW TABLES IN silver

database,tableName,isTemporary
silver,agencia,false
silver,apolice,false
silver,carro,false
silver,categoria,false
silver,cidade,false
silver,cliente,false
silver,endereco,false
silver,estado,false
silver,marca,false
silver,modelo,false


In [0]:
%sql
DESCRIBE DETAIL silver.reserva

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9084b7d8-a474-4489-aa86-39b11a694a77,workspace.silver.reserva,null,,2026-05-13T16:53:18.429Z,2026-05-13T16:53:21.000Z,List(),List(),1,4060,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE EXTENDED silver.reserva

col_name,data_type,comment
CODIGO_RESERVA,int,null
CODIGO_CLIENTE,int,null
CODIGO_CARRO,int,null
CODIGO_AGENCIA,int,null
DATA_RETIRADA,date,null
DATA_DEVOLUCAO,date,null
VALOR_DIARIA,double,null
VALOR_TOTAL,double,null
STATUS_RESERVA,string,null
NOME_TABELA_BRONZE,string,null
